# 7 · Shared references (produce / consume)

An edge is one way to wire node outputs to inputs. A **shared reference** is the other way: a producer marks an output as shareable, and any consumer anywhere in the graph can bind an input to it — including *inside* compound regions like for-each loops.

Two use cases drive this:

- **Fan-out without N edges.** One node computes a value (e.g. a pseudonymisation map); many downstream nodes consume it.
- **Cross-region binding.** A system prompt defined once at the top of a flow, consumed by an LLM node inside a for-each that iterates over 1,000 records.

Producers opt in per instance via `produces`. Consumers bind by identity via `consumes`. Both are optional fields on `GraphNode`; a graph that uses neither is unchanged.

In [ ]:
from typing import Annotated

from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.compound.for_each import FOR_EACH
from conductor.errors import CompilationError
from conductor.execution.engine import collect, execute
from conductor.types import NodeCategory
from conductor.widgets import ConnectionList, Output, Text

registry = NodeRegistry()

## A small node library

- `build-map` emits a `{name: pseudonym}` mapping.
- `redact` rewrites a text using the mapping.
- Standard `for-each-start` / `for-each-end` markers for the loop example below.

In [ ]:
@registry.node("build-map", version=1, name="Build Map", description="Builds a pseudonymisation map")
def build_map(
    seed: Annotated[str, Text(label="Seed")] = "0",
) -> Annotated[dict[str, str], Output(label="Mapping")]:
    return {"Alice": f"P001-{seed}", "Bob": f"P002-{seed}", "Carol": f"P003-{seed}"}


@registry.node("redact", version=1, name="Redact", description="Redacts names using a mapping")
def redact(
    text: Annotated[str, Text(label="Text")],
    mapping: Annotated[dict[str, str], Text(label="Mapping")],
) -> Annotated[str, Output(label="Redacted")]:
    out = text
    for k, v in mapping.items():
        out = out.replace(k, v)
    return out


@registry.node(
    "for-each-start", version=1,
    name="For Each (Start)", description="Iterates",
    category=NodeCategory.CONTROL,
)
def for_each_start(
    items: Annotated[list[str], ConnectionList(label="Items")],
) -> tuple[
    Annotated[str, Output(label="Item")],
    Annotated[int, Output(label="Index")],
]:
    raise NotImplementedError


@registry.node(
    "for-each-end", version=1,
    name="For Each (End)", description="Collects",
    category=NodeCategory.CONTROL,
)
def for_each_end(
    item: Annotated[str, Text(label="Item")],
) -> Annotated[list[str], Output(label="Collected")]:
    raise NotImplementedError


@registry.node("constant-text", version=1, name="Constant", description="Emits a fixed string")
def constant_text(
    value: Annotated[str, Text(label="Value")] = "",
) -> Annotated[str, Output(label="Value")]:
    return value


@registry.node(
    "redact-with-prefix", version=1,
    name="Redact with Prefix", description="Prefix + redact — used in the loop demo",
)
def redact_with_prefix(
    text: Annotated[str, Text(label="Text")],
    mapping: Annotated[dict[str, str], Text(label="Mapping")],
    prefix: Annotated[str, Text(label="Prefix")],
) -> Annotated[str, Output(label="Out")]:
    out = text
    for k, v in mapping.items():
        out = out.replace(k, v)
    return f"{prefix}: {out}" 

## Example 1 — basic produce / consume (no edge needed)

`mapper` produces its `result` as a shared reference. `red` consumes it on its `mapping` input. No `GraphEdge` is required; the compiler turns the consume binding into a scheduling dependency so `mapper` runs first.

In [ ]:
nodes = [
    GraphNode(
        "mapper", "build-map@1", {"seed": "x"},
        produces={"result": "pseudonym map"},
    ),
    GraphNode(
        "red", "redact@1", {"text": "Alice met Bob."},
        consumes={"mapping": ("mapper", "result")},
    ),
]

compiled = compile(nodes=nodes, edges=[], registry=registry)

print("execution_order:", compiled.execution_order)
print("consume_map:   ", compiled.consume_map)
print("edge_map:      ", compiled.edge_map)

results = await collect(execute(compiled))
print()
print("redacted:      ", results["red"]["result"])

## Example 2 — fan-out

One producer, three consumers. Still zero edges.

In [ ]:
nodes = [
    GraphNode("mapper", "build-map@1", {"seed": "y"}, produces={"result": "pseudonym map"}),
    GraphNode("r1", "redact@1", {"text": "Alice spoke."},
              consumes={"mapping": ("mapper", "result")}),
    GraphNode("r2", "redact@1", {"text": "Bob replied."},
              consumes={"mapping": ("mapper", "result")}),
    GraphNode("r3", "redact@1", {"text": "Carol nodded."},
              consumes={"mapping": ("mapper", "result")}),
]

compiled = compile(nodes=nodes, edges=[], registry=registry)
results = await collect(execute(compiled))

for rid in ("r1", "r2", "r3"):
    print(f"{rid}: {results[rid]['result']}")

## Example 3 — broadcast into a for-each loop

The flagship use case. A top-level producer feeds a node that lives **inside** the for-each body. No edge can cross the region boundary — but a consume binding can. The same producer value is read on every iteration (broadcast, not iteration).

In [ ]:
nodes = [
    GraphNode("mapper", "build-map@1", {"seed": "z"}, produces={"result": "pseudonym map"}),
    GraphNode(
        "sys", "constant-text@1", {"value": "[SYSTEM]"},
        produces={"result": "system prefix"},
    ),
    GraphNode("start", "for-each-start@1", {"items": [
        "Alice called.",
        "Bob answered.",
        "Carol waited.",
    ]}),
    GraphNode(
        "body", "redact-with-prefix@1", None,
        consumes={
            "mapping": ("mapper", "result"),   # dict, shared across every iteration
            "prefix": ("sys", "result"),       # string, shared across every iteration
        },
    ),
    GraphNode("end", "for-each-end@1", None),
]
edges = [
    GraphEdge("e1", "start", "body", "output_1", "text"),
    GraphEdge("e2", "body", "end", "result", "item"),
]

compiled = compile(nodes=nodes, edges=edges, registry=registry, compound_types=[FOR_EACH])
results = await collect(execute(compiled))

for line in results["end"]["result"]:
    print(line)

## Example 4 — compile-time validation

Consume bindings are validated before any node runs. A few common mistakes and the errors they raise.

In [ ]:
# 4a — consumer references an unpublished output
try:
    compile(
        nodes=[
            GraphNode("p", "build-map@1", {"seed": "q"}),  # no `produces`
            GraphNode("c", "redact@1", {"text": "Alice"},
                      consumes={"mapping": ("p", "result")}),
        ],
        edges=[],
        registry=registry,
    )
except CompilationError as e:
    print("4a:", e)

In [ ]:
# 4b — edge AND consume targeting the same input handle
try:
    compile(
        nodes=[
            GraphNode("p1", "build-map@1", {"seed": "a"}, produces={"result": "m"}),
            GraphNode("p2", "build-map@1", {"seed": "b"}, produces={"result": "m2"}),
            GraphNode("c", "redact@1", {"text": "Alice"},
                      consumes={"mapping": ("p1", "result")}),
        ],
        edges=[GraphEdge("e1", "p2", "c", "result", "mapping")],  # collides with consume
        registry=registry,
    )
except CompilationError as e:
    print("4b:", e)

In [ ]:
# 4c — cycle formed purely by consume bindings
try:
    compile(
        nodes=[
            GraphNode("a", "redact@1", {"text": "x"},
                      produces={"result": "ra"},
                      consumes={"mapping": ("b", "result")}),
            GraphNode("b", "redact@1", {"text": "y"},
                      produces={"result": "rb"},
                      consumes={"mapping": ("a", "result")}),
        ],
        edges=[],
        registry=registry,
    )
except Exception as e:
    print("4c:", type(e).__name__, "-", e)

## Resolution precedence

When an input could be resolved from multiple sources, the resolver uses this order (first match wins):

1. **Explicit edge** targeting the input
2. **Consume binding** (shared reference)
3. **Static data** on the node
4. **Widget default** (Pydantic)

Steps 1 and 2 cannot both apply to the same input handle — the compiler rejects that at compile time (see 4b above). Step 2 **does** override static data: if you set `data={"text": "x"}` on a node *and* add a consume binding on `text`, the consume value wins.

## Where to go next

- `docs/shared-references.md` — the full v1 design spec.
- `tests/test_core/test_shared_references.py` — 29 behavioral tests covering every rule.
- **Host apps** implementing the UI: render a "Share output as …" toggle per output handle (per instance, not per node type), and a "Bind to shared reference" dropdown per input handle that lists every type-compatible producer currently on the canvas. Reference identity is the `(producer_node_id, output_handle)` tuple; the display label is free-form.